# Segundo avance — Clasificación de tumores de mama
**Universidad de la Costa | Minería de Datos | Periodo 2**

**Tipo de problema:** clasificación supervisada con dos clases (maligno o benigno).

**Pregunta que queremos responder:** ¿Qué tanto sirven las características de las células del dataset Breast Cancer Wisconsin (Diagnostic) para saber si un tumor es maligno o benigno, usando regresión logística y un árbol de decisión?

**Fuente de los datos:** [scikit-learn Breast Cancer Wisconsin](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html). Seguimos con el mismo dataset, el mismo objetivo y el mismo enfoque del primer avance. Aclaramos que esto es un ejercicio académico y no una validación médica.

## 1. Dataset
**Nombre:** Breast Cancer Wisconsin (Diagnostic). **Fuente:** [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).

**Tamaño esperado:** 569 filas y 31 columnas (30 variables para predecir y la columna objetivo, `target`).

**Qué vamos a hacer:** explorar las medidas de las células de cada muestra y mirar cómo se relacionan con el diagnóstico (0 = maligno, 1 = benigno). El dataset ya viene con scikit-learn, así que no hay que descargar nada aparte.

In [ ]:
# Cargamos las librerías y el dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

bunch = load_breast_cancer(as_frame=True)
df = bunch.frame.copy()
print("Dimensiones:", df.shape)
display(df.head())

## 2. Variables
Las 30 variables para predecir son medidas numéricas sacadas de imágenes de células. Cada una aparece en tres versiones: **mean** (promedio), **error** (error estándar) y **worst** (el peor valor, es decir, el más grande). Se miden estas diez características: radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal.

La variable que queremos predecir es `target`.

In [ ]:
# Armamos una tablita con el nombre y el tipo de cada columna
diccionario = pd.DataFrame({"columna":df.columns, "tipo":df.dtypes.astype(str).values})
display(diccionario)
print("Objetivo:", dict(enumerate(bunch.target_names)))

## 3. Exploración inicial
Primero revisamos cómo es el dataset: cuántas filas y columnas tiene, qué tipo de datos son y un resumen de estadísticas.

In [ ]:
# Vemos tamaño, tipos de datos y estadísticas básicas
print("Filas y columnas:",df.shape)
df.info()
display(df.describe().T)
numericas = df.select_dtypes(include="number").columns.tolist()
categoricas = df.select_dtypes(exclude="number").columns.tolist()
print("Numéricas:",len(numericas),"Categóricas:",categoricas)

## 4. Revisión de la calidad de los datos
Antes de usar los datos, revisamos que estén bien: si hay datos faltantes, filas repetidas, valores infinitos, que el objetivo tenga solo los códigos que esperamos, y si hay valores extremos (con el método IQR).

Los valores extremos **no los borramos** de una vez, porque podrían ser casos reales de pacientes y no errores.

In [ ]:
# Revisamos nulos, duplicados, etiquetas, infinitos y valores extremos (IQR)
print("Nulos:",int(df.isna().sum().sum()))
print("Duplicados completos:",int(df.duplicated().sum()))
print("Etiquetas:",df.target.value_counts().to_dict())
print("Infinitos:",int(np.isinf(df.select_dtypes(include="number").to_numpy()).sum()))
q1=df.drop(columns="target").quantile(.25);q3=df.drop(columns="target").quantile(.75);iqr=q3-q1
fuera=((df.drop(columns="target") < (q1-1.5*iqr)) | (df.drop(columns="target") > (q3+1.5*iqr)))
print("Registros con al menos un extremo IQR:",int(fuera.any(axis=1).sum()))
print("Columnas constantes:",df.columns[df.nunique(dropna=False)<=1].tolist())

## 5. Limpieza inicial
Lo que hicimos fue:
- Arreglar los nombres de las columnas (todo en minúscula y con guiones bajos).
- Quitar filas completamente repetidas, si aparecían.
- Comprobar que `target` siguiera teniendo sus dos códigos.

No hubo que rellenar datos porque no faltaba ninguno, y tampoco quitamos valores extremos. Además guardamos una copia del dataset original para poder comparar después.

In [ ]:
# Limpiamos nombres de columnas y guardamos el CSV limpio
original=df.copy()
limpio=df.copy()
limpio.columns=(limpio.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+","_",regex=True).str.strip("_"))
limpio=limpio.drop_duplicates().reset_index(drop=True)
assert set(limpio.target.unique())=={0,1}
assert limpio.isna().sum().sum()==0
print("Antes:",original.shape,"Después:",limpio.shape)
limpio.to_csv("datos_cancer_mama_limpios.csv",index=False)

## 6. Gráficos e interpretación

In [ ]:
# Gráfico de barras: cuántos casos hay de cada diagnóstico
conteos=limpio.target.map({0:"Maligno",1:"Benigno"}).value_counts()
ax=conteos.plot(kind="bar",title="Distribución de diagnósticos",rot=0)
ax.set_ylabel("Registros");plt.tight_layout();plt.show()
print(conteos.to_dict())

**Interpretación 1:** hay más casos benignos que malignos. Esta diferencia hay que tenerla en cuenta cuando escojamos las métricas para evaluar los modelos.

In [ ]:
# Boxplot del radio medio según el diagnóstico
limpio.boxplot(column="mean_radius",by="target")
plt.suptitle("");plt.title("Radio medio según diagnóstico (0 maligno, 1 benigno)")
plt.xlabel("Diagnóstico");plt.ylabel("Radio medio");plt.tight_layout();plt.show()

**Interpretación 2:** el radio medio es distinto entre tumores malignos y benignos, pero las cajas se solapan un poco. Entonces, el radio solo no alcanza para decidir el diagnóstico.

In [ ]:
# Dispersión: radio medio vs. textura media
plt.figure(figsize=(7,5))
sns.scatterplot(data=limpio,x="mean_radius",y="mean_texture",hue="target",palette="Set2",alpha=.7)
plt.title("Radio medio frente a textura media");plt.tight_layout();plt.show()

**Interpretación 3:** en el gráfico de puntos se ven zonas donde se juntan los casos de cada clase y otras donde se mezclan. Por eso hay que usar varias variables al tiempo para poder predecir mejor.

## 7. Registro del uso de inteligencia artificial

- **Herramienta:** ChatGPT (OpenAI).
- **¿Para qué la usamos?** La usamos para tener una guía al organizar la exploración de los datos, revisar qué aspectos debíamos comprobar y pensar en gráficos que nos ayudaran a entender la información.
- **Prompt utilizado:** «Estoy haciendo un proyecto de Minería de Datos sobre la clasificación de tumores de mama. Ayúdame a organizar la primera parte del trabajo usando el conjunto de datos Breast Cancer Wisconsin de Python. Explícame de manera sencilla cómo revisar los datos, detectar valores faltantes o repetidos y hacer gráficos que permitan comparar los casos benignos y malignos. No inventes resultados y explícame qué debo comprobar ejecutando el código».
- **¿Qué revisamos por nuestra cuenta?** Comprobamos los nombres y las clases del conjunto de datos con la documentación de scikit-learn. También ejecutamos el notebook, revisamos los conteos y miramos los gráficos para confirmar que tuvieran sentido.
- **¿Qué aprendimos?** La IA nos dio ideas para organizar el trabajo, pero tuvimos que revisar cada resultado. Por ejemplo, no eliminamos los valores extremos automáticamente porque podían representar diferencias reales entre las muestras.
- **Nota:** este texto es un borrador y el grupo debe confirmar que el prompt y las actividades descritas coincidan con lo que realmente hizo.

## 8. Conclusión inicial
Con la primera revisión vimos que el dataset está ordenado: tiene 569 registros, 30 variables para predecir y una etiqueta con dos clases. En la versión de scikit-learn no se esperan datos faltantes ni filas repetidas; el código lo comprueba y además cuenta cuántos valores extremos hay con el método IQR. Arreglamos los nombres de las columnas y guardamos una copia limpia sin quitar los valores extremos.

En el segundo periodo vamos a dividir los datos en entrenamiento y prueba y comparar dos clasificadores. Vamos a fijarnos en la sensibilidad (recall), la precisión y la matriz de confusión, y a validar bien para que no se filtre información de la prueba al entrenamiento.

## 9. Continuidad y preparación para los modelos
Seguimos con los 569 registros y las 30 medidas como variables para predecir, más la etiqueta `target` (0 = maligno, 1 = benigno). No usamos identificadores. Mantenemos la revisión de nulos, duplicados y valores extremos, y los extremos siguen sin eliminarse porque pueden ser diferencias reales entre pacientes.

Para evitar que se filtre información, la estandarización se calcula solo con los datos de entrenamiento usando un `Pipeline`.

Otro cambio: para evaluar, volvemos a codificar la variable objetivo con **1 = maligno**. Así la precisión y el recall hablan del caso que más importa, que es encontrar los tumores malignos.

In [ ]:
# Importamos lo de modelos y separamos X (variables) y y (1 = maligno)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score
from IPython.display import display

X = limpio.drop(columns=["target"]).copy()
y = (limpio["target"] == 0).astype(int)  # 1 = maligno
print("Datos finales (filas, predictores):", X.shape)
print("Nulos:", int(X.isna().sum().sum()), "Duplicados:", int(limpio.duplicated().sum()))
print("Clases (0 benigno, 1 maligno):", y.value_counts().sort_index().to_dict())

## 10. Exploración ampliada (EDA)
Aquí miramos con más detalle cómo se distribuyen las variables según el diagnóstico, cómo se correlacionan entre sí y si hay valores extremos. Recordamos que una relación entre variables no quiere decir que una cause a la otra. Los gráficos del primer avance se dejan arriba como evidencia de que seguimos el mismo trabajo.

In [ ]:
# Histograma, boxplot, correlaciones y heatmap
fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.histplot(data=limpio,x="mean_radius",hue="target",kde=True,ax=axes[0],element="step",common_norm=False)
axes[0].set(title="Figura 4. Radio medio por diagnóstico",xlabel="Radio medio",ylabel="Frecuencia")
sns.boxplot(data=limpio,x="target",y="worst_area",ax=axes[1])
axes[1].set(title="Figura 5. Área máxima según diagnóstico",xlabel="Diagnóstico (0 maligno, 1 benigno)",ylabel="Área máxima")
plt.tight_layout();plt.show()
correlaciones=limpio.corr(numeric_only=True)["target"].drop("target").sort_values()
print("Variables con asociaciones absolutas más altas con target:")
display(correlaciones.reindex(correlaciones.abs().sort_values(ascending=False).index).head(10).to_frame("Correlación de Pearson"))
plt.figure(figsize=(9,5));sns.heatmap(limpio[["mean_radius","mean_perimeter","mean_area","worst_radius","worst_area","target"]].corr(),annot=True,cmap="coolwarm",vmin=-1,vmax=1);plt.title("Figura 6. Correlaciones de variables seleccionadas");plt.tight_layout();plt.show()

**Interpretación del EDA ampliado:** el radio medio y el área máxima (`worst_area`) son distintos entre diagnósticos, aunque hay zonas donde se mezclan. Las variables de tamaño (radio, perímetro y área) están muy correlacionadas entre sí, y eso puede complicar la lectura de los coeficientes del modelo. Dejamos los valores extremos porque no hay evidencia de que sean errores de captura. Para manejar que hay más benignos que malignos, dividimos los datos de forma estratificada y miramos las métricas por clase.

## 11. Modelos supervisados y evaluación
Dividimos los datos con una partición estratificada de 80% para entrenar y 20% para probar (`random_state=42`, para que el resultado se pueda repetir).

- **Regresión logística:** estandariza los datos dentro del pipeline.
- **Árbol de decisión:** no necesita estandarizar.

Los dos modelos se entrenan solo con los datos de entrenamiento. Los comparamos con accuracy, precision, recall, F1 y AUC. Como la clase positiva es **maligno**, los falsos negativos son lo más grave (un tumor maligno que el modelo dice que es benigno). Estas métricas no demuestran que el modelo sirva en la práctica médica.

In [ ]:
# Dividimos 80/20, entrenamos los dos modelos y sacamos las métricas
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
modelos={
 "Regresión logística":Pipeline([("escalado",StandardScaler()),("modelo",LogisticRegression(max_iter=2000,random_state=42))]),
 "Árbol de decisión":DecisionTreeClassifier(max_depth=4,random_state=42)
}
resultados=[]; matrices={}
for nombre,modelo in modelos.items():
    modelo.fit(X_train,y_train)
    pred=modelo.predict(X_test)
    score=modelo.predict_proba(X_test)[:,1]
    resultados.append({"Modelo":nombre,"Accuracy":accuracy_score(y_test,pred),"Precision_maligno":precision_score(y_test,pred,zero_division=0),"Recall_maligno":recall_score(y_test,pred,zero_division=0),"F1_maligno":f1_score(y_test,pred,zero_division=0),"AUC":roc_auc_score(y_test,score),"FN_malignos":int(((y_test==1)&(pred==0)).sum()),"FP_malignos":int(((y_test==0)&(pred==1)).sum())})
    matrices[nombre]=confusion_matrix(y_test,pred,labels=[0,1])
    print("\n",nombre,"\n",classification_report(y_test,pred,target_names=["Benigno","Maligno"],zero_division=0))
    ConfusionMatrixDisplay(matrices[nombre],display_labels=["Benigno","Maligno"]).plot(cmap="Blues",values_format="d")
    plt.title("Matriz de confusión: "+nombre);plt.show()
resumen=pd.DataFrame(resultados).set_index("Modelo")
display(resumen.round(4))
print("Entrenamiento:",len(X_train),"Prueba:",len(X_test))

In [ ]:
# Vemos qué modelo deja pasar menos tumores malignos
print("Modelo con menor cantidad de falsos negativos:", resumen.sort_values(["FN_malignos","Recall_maligno","F1_maligno"],ascending=[True,False,False]).index[0])
print("Interpretación: FN_malignos = tumores malignos clasificados como benignos. La comparación corresponde a una única partición; no es una validación externa.")

## 12. Conclusiones preliminares y limitaciones
Los números y las matrices de confusión de arriba son la evidencia de este corte. Comparamos dos modelos que funcionan distinto y le dimos más importancia a cuántos tumores malignos no se detectan, junto con el recall y el F1 de la clase maligna.

Limitaciones: la muestra es pequeña y viene de un lugar específico, y los resultados cambian según cómo se haga la partición. Por eso no podemos decir que funcionaría igual en hospitales ni con población colombiana.

Para el tercer corte recomendamos usar validación cruzada para ver si los resultados son estables, analizar en qué casos se equivocan los modelos y discutir los sesgos y si realmente se podría aplicar.

## 13. Declaración y revisión del uso de inteligencia artificial

**Herramienta utilizada:** ChatGPT (OpenAI).

**¿Para qué la utilizamos?** La usamos como apoyo para continuar el proyecto, organizar el código de los modelos y entender mejor algunos resultados. La herramienta nos ayudó a tener una guía, pero no reemplazó el trabajo de revisar y ejecutar el programa.

**Prompt utilizado:**

«Estoy realizando un proyecto de Minería de Datos sobre la clasificación de tumores de mama como benignos o malignos. Ya tengo un notebook con la exploración de los datos. Ayúdame a continuar el trabajo comparando regresión logística y árbol de decisión, usando Python y scikit-learn. Quiero que revises la división entre datos de entrenamiento y prueba, las métricas y la validación cruzada. Explícame el código con palabras sencillas y ten cuidado de no inventar resultados. También revisa que el escalado se haga correctamente y que se tenga en cuenta que los falsos negativos de la clase maligna son importantes».

**¿Cómo verificamos la respuesta?**

No nos quedamos solamente con lo que respondió ChatGPT. Ejecutamos el notebook y revisamos que el código terminara sin errores. Comprobamos que el conjunto de datos tuviera las columnas esperadas y que la variable objetivo estuviera codificada correctamente. También revisamos que la estandarización se hiciera dentro del `Pipeline`, para evitar usar información de los datos de prueba durante el entrenamiento.

Después miramos las métricas que produjo el código, como accuracy, precision, recall, F1 y AUC, y revisamos las matrices de confusión. Así pudimos comparar los dos modelos con los mismos datos. También tuvimos en cuenta los falsos negativos, porque representan casos malignos que el modelo podría clasificar como benignos.

**¿Qué debemos tener presente?** Aunque los resultados se vean buenos, eso no demuestra que el modelo pueda utilizarse para diagnosticar pacientes. El conjunto de datos es limitado y se necesitarían más pruebas antes de pensar en un uso real.

**Pendiente del grupo:** confirmar que este prompt representa la consulta que se hizo realmente, revisar los resultados con la ejecución final y completar cualquier decisión que haya tomado el grupo.